In [ ]:
# ============================================
# Cell 1: 初始化和配置加载
# ============================================

# 设置工作目录
setwd("/home/ailab/caohao/AdaDiss/")

# 加载配置文件
source("config.R")
source("utils.R")
source("visualization.R")
source("export.R")

# 验证配置
validate_config()

# 创建输出目录
create_output_dirs()

# 设置随机种子
set.seed(params$seed)

# 加载必要的包
required_packages <- c(
    "Seurat", "BPCells", "SeuratObject", "SeuratDisk",
    "tidyverse", "jsonlite", "ggplot2", "ggpmisc",
    "scales", "cowplot", "gridExtra", "viridis", "hrbrthemes"
)

load_packages(required_packages)

# 设置Seurat内存限制
options(future.globals.maxSize = 1e9)

# 打印配置信息
print_config()

In [ ]:
# ============================================
# Cell 2: 加载Flex参考数据
# ============================================

process_flex_data <- function() {
    cat("  步骤1: 加载原始数据...\n")
    flex_obj <- Read10X_h5(paths$flex_h5) %>% CreateSeuratObject()
    
    cat("  步骤2: 配置BPCells磁盘存储...\n")
    if (!dir.exists(paths$flex_bpcells_dir)) {
        write_matrix_dir(mat = flex_obj[["RNA"]]$counts, dir = paths$flex_bpcells_dir)
    }
    flex_obj[['RNA']]$counts <- open_matrix_dir(dir = paths$flex_bpcells_dir)
    
    cat("  步骤3: 计算线粒体比例...\n")
    flex_obj[["percent.mt"]] <- PercentageFeatureSet(flex_obj, pattern = "^MT-")
    
    cat("  步骤4: QC过滤...\n")
    n_before <- ncol(flex_obj)
    flex_obj <- subset(flex_obj, 
                       subset = nCount_RNA > params$flex_min_counts & 
                                nCount_RNA < params$flex_max_counts & 
                                percent.mt < params$flex_max_mt)
    cat("    过滤前:", n_before, "细胞 → 过滤后:", ncol(flex_obj), "细胞\n")
    
    cat("  步骤5: 添加细胞类型注释...\n")
    if (file.exists(paths$flex_annotation)) {
        ann_df <- safe_read_csv(paths$flex_annotation)
        cell_types <- setNames(ann_df$Cell.Annotation, ann_df$Barcode)
        cell_types <- cell_types[names(cell_types) %in% colnames(flex_obj)]
        flex_obj <- AddMetaData(flex_obj, cell_types[colnames(flex_obj)], "cell_type")
        cat("    注释了", length(unique(flex_obj$cell_type)), "种细胞类型\n")
    } else {
        stop("找不到注释文件: ", paths$flex_annotation)
    }
    
    return(flex_obj)
}

# 加载或处理Flex数据
flex_data.obj <- time_it(function() {
    load_or_process(paths$flex_cache, process_flex_data, verbose = params$verbose)
})

# 检查数据完整性
check_data_integrity(flex_data.obj, "Flex数据处理", expected_min_cells = 1000)

cat("\n📊 Flex数据摘要:\n")
cat("  - 细胞数量:", format_number(ncol(flex_data.obj)), "\n")
cat("  - 基因数量:", format_number(nrow(flex_data.obj)), "\n")
cat("  - 细胞类型:", length(unique(flex_data.obj$cell_type)), "\n")

In [ ]:
# ============================================
# Cell 3: 加载Xenium数据
# ============================================

process_xenium_data <- function() {
    cat("  步骤1: 加载原始数据...\n")
    xenium_obj <- LoadXenium(paths$xenium_dir, fov = "fov", molecule.coordinates = FALSE)
    DefaultAssay(xenium_obj) <- "Xenium"
    
    cat("  步骤2: 配置BPCells磁盘存储...\n")
    if (!dir.exists(paths$xenium_bpcells_dir)) {
        write_matrix_dir(mat = xenium_obj[["Xenium"]]$counts, dir = paths$xenium_bpcells_dir)
    }
    xenium_obj[['Xenium']]$counts <- open_matrix_dir(dir = paths$xenium_bpcells_dir)
    
    cat("  步骤3: 过滤空细胞...\n")
    n_before <- ncol(xenium_obj)
    xenium_obj <- subset(xenium_obj, subset = nCount_Xenium > 0)
    cat("    过滤前:", n_before, "细胞 → 过滤后:", ncol(xenium_obj), "细胞\n")
    
    cat("  步骤4: 添加log转换元数据...\n")
    xenium_obj@meta.data$nCount_Xenium_log <- log1p(xenium_obj@meta.data$nCount_Xenium)
    xenium_obj@meta.data$nFeature_Xenium_log <- log1p(xenium_obj@meta.data$nFeature_Xenium)
    
    return(xenium_obj)
}

# 加载或处理Xenium数据
xenium.obj <- time_it(function() {
    load_or_process(paths$xenium_cache, process_xenium_data, verbose = params$verbose)
})

# 检查数据完整性
check_data_integrity(xenium.obj, "Xenium数据处理", expected_min_cells = 1000)

cat("\n📊 Xenium数据摘要:\n")
cat("  - 细胞数量:", format_number(ncol(xenium.obj)), "\n")
cat("  - 基因数量:", format_number(nrow(xenium.obj)), "\n")

In [ ]:
# ============================================
# Cell 4: 基因表达相关性分析（可选）
# ============================================

if (file.exists(paths$xenium_gene_panel)) {
    cat("📈 计算基因表达相关性...\n")
    
    get_gex_means <- function(xenium_obj, flex_obj) {
        xen_means <- data.frame(
            mean_counts = rowMeans(xenium_obj[["Xenium"]]$counts),
            gene = rownames(xenium_obj[["Xenium"]]$counts)
        ) %>% arrange(desc(mean_counts)) %>% mutate(Rank = 1:n())
        
        flex_means <- data.frame(
            mean_counts = rowMeans(flex_obj[["RNA"]]$counts),
            gene = rownames(flex_obj[["RNA"]]$counts)
        ) %>% arrange(desc(mean_counts)) %>% mutate(Rank = 1:n())
        
        return(merge(xen_means, flex_means, by.x = "gene", by.y = "gene", all.x = TRUE))
    }
    
    gene_panel <- fromJSON(paths$xenium_gene_panel)
    targets <- gene_panel$payload$targets
    panel_source <- setNames(
        data.frame(cbind(targets$source$identity$name, targets$type$data$name)), 
        c("gene_panel", "gene")
    )
    
    merged_means <- get_gex_means(xenium.obj, flex_data.obj)
    merged_means <- merge(merged_means, panel_source, by.x = "gene", by.y = "gene", all.x = TRUE) %>%
                   na.omit() %>% arrange(gene_panel)
    
    # 创建相关性图
    cor_plot <- create_correlation_plot(
        merged_means, 
        x_col = "mean_counts.y", 
        y_col = "mean_counts.x",
        color_col = "gene_panel",
        title = "Gene Expression Correlation"
    )
    
    print(cor_plot)
    cat("✅ 相关性分析完成\n")
} else {
    cat("⚠️ 跳过相关性分析（找不到gene_panel.json）\n")
}

In [ ]:
# ============================================
# Cell 5: Flex参考数据处理
# ============================================

cat("🔧 处理Flex参考数据...\n")

DefaultAssay(flex_data.obj) <- "RNA"

flex_data.obj <- time_it(function() {
    flex_data.obj %>%
        NormalizeData() %>%
        FindVariableFeatures() %>%
        ScaleData() %>%
        RunPCA(verbose = FALSE) %>%
        RunUMAP(dims = 1:15, verbose = FALSE) %>%
        FindNeighbors(dims = 1:15) %>%
        FindClusters(resolution = 0.5)
})

# 创建可视化
flex_plots <- list(
    clusters = create_dim_plot(flex_data.obj, group_by = "RNA_snn_res.0.5", 
                               title = "Flex Clusters", pt_size = 0.2),
    cell_types = create_dim_plot(flex_data.obj, group_by = "cell_type", 
                                 title = "Flex Cell Types", pt_size = 0.2)
)

# 显示组合图
combined_flex <- create_composite_plot(flex_plots, ncol = 2)
print(combined_flex)

cat("✅ Flex数据处理完成\n")

In [ ]:
# ============================================
# Cell 6: Xenium数据降维和聚类
# ============================================

cat("🔧 处理Xenium数据（细胞数:", format_number(ncol(xenium.obj)), "）\n")

DefaultAssay(xenium.obj) <- "Xenium"

xenium.obj <- time_it(function() {
    xenium.obj %>%
        NormalizeData() %>%
        FindVariableFeatures() %>%
        ScaleData() %>%
        RunPCA(npcs = params$xenium_npcs, verbose = FALSE) %>%
        RunUMAP(dims = params$xenium_dims, verbose = FALSE) %>%
        FindNeighbors(reduction = "pca", dims = params$xenium_dims) %>%
        FindClusters(resolution = params$xenium_resolution, 
                    cluster.name = params$xenium_cluster_name)
})

# 创建可视化
xenium_plots <- list(
    clusters_umap = create_dim_plot(xenium.obj, group_by = params$xenium_cluster_name,
                                    title = "Xenium Clusters"),
    clusters_spatial = create_spatial_plot(xenium.obj, group_by = params$xenium_cluster_name,
                                           title = "Xenium Spatial Clusters")
)

# 显示图表
print(xenium_plots$clusters_umap)
if (!is.null(xenium_plots$clusters_spatial)) {
    print(xenium_plots$clusters_spatial)
}

# 聚类统计
cat("\n聚类统计:\n")
cluster_counts <- sort(table(xenium.obj$clusters), decreasing = TRUE)
print(cluster_counts)

cat("✅ Xenium数据处理完成\n")

In [ ]:
# ============================================
# Cell 7: 标签转移（核心步骤）
# ============================================

if (!"predicted.id" %in% colnames(xenium.obj@meta.data)) {
    
    cat("🏷️  开始标签转移...\n")
    
    # 获取共同基因
    flex_xen_common_genes <- intersect(rownames(xenium.obj), rownames(flex_data.obj))
    cat("  共同基因数量:", length(flex_xen_common_genes), "\n")
    
    if (length(flex_xen_common_genes) < 100) {
        warning("共同基因数量较少（<100），标签转移效果可能不佳")
    }
    
    # 创建Flex子集
    cat("  创建Flex子集...\n")
    flex_subset <- CreateSeuratObject(
        counts = flex_data.obj[["RNA"]]$counts[flex_xen_common_genes,],
        meta = flex_data.obj@meta.data
    ) %>%
        NormalizeData() %>%
        FindVariableFeatures() %>%
        ScaleData() %>%
        RunPCA(verbose = FALSE)
    
    # 准备数据
    cat("  准备数据...\n")
    flex_data.obj[["RNA"]]$counts <- as(flex_data.obj[["RNA"]]$counts, "dgCMatrix")
    xenium.obj[["Xenium"]]$counts <- as(xenium.obj[["Xenium"]]$counts, "dgCMatrix")
    
    # 寻找锚点
    cat("  寻找锚点（这可能需要一些时间）...\n")
    anchors_from_flex <- time_it(function() {
        FindTransferAnchors(
            reference = flex_subset,
            query = xenium.obj,
            features = flex_xen_common_genes,
            dims = params$transfer_dims,
            reference.reduction = "pca"
        )
    })
    
    # 转移标签
    cat("  转移标签...\n")
    label_transfer <- time_it(function() {
        TransferData(
            anchorset = anchors_from_flex,
            refdata = flex_subset$cell_type,
            dims = params$transfer_dims
        )
    })
    
    # 添加预测结果
    xenium.obj <- AddMetaData(xenium.obj, metadata = label_transfer)
    
    cat("✅ 标签转移完成\n")
    cat("\n预测细胞类型分布:\n")
    print(table(xenium.obj$predicted.id))
    
    # 显示预测分数统计
    if ("predicted.id.score" %in% colnames(xenium.obj@meta.data)) {
        cat("\n预测分数统计:\n")
        print(summary(xenium.obj$predicted.id.score))
    }
    
} else {
    cat("✅ 标签转移结果已存在，跳过\n")
}

In [ ]:
# ============================================
# Cell 8: 可视化标签转移结果
# ============================================

if ("predicted.id" %in% colnames(xenium.obj@meta.data)) {
    
    cat("📊 绘制预测结果...\n")
    
    # 添加预测分数（如果不存在）
    if (!"prediction_score" %in% colnames(xenium.obj@meta.data)) {
        score_cols <- grep("prediction\\.score\\.", colnames(xenium.obj@meta.data), value = TRUE)
        
        if (length(score_cols) > 0) {
            xenium.obj@meta.data$prediction_score <- sapply(1:nrow(xenium.obj@meta.data), function(i) {
                cell_type <- xenium.obj@meta.data$predicted.id[i]
                if (!is.na(cell_type) && cell_type %in% names(reverse_mapping)) {
                    score_col <- paste0("prediction.score.", reverse_mapping[cell_type])
                    if (score_col %in% colnames(xenium.obj@meta.data)) {
                        return(as.numeric(xenium.obj@meta.data[i, score_col]))
                    }
                }
                return(NA_real_)
            })
        }
    }
    
    # 生成所有可视化
    prediction_plots <- generate_visualizations(
        xenium.obj, 
        flex_data.obj, 
        paths$output_plots,
        params
    )
    
    # 显示关键图表
    if (!is.null(prediction_plots$xenium_predicted_umap)) {
        print(prediction_plots$xenium_predicted_umap)
    }
    
    if (!is.null(prediction_plots$xenium_predicted_spatial)) {
        print(prediction_plots$xenium_predicted_spatial)
    }
    
    if (!is.null(prediction_plots$score_violin)) {
        print(prediction_plots$score_violin)
    }
    
    if (!is.null(prediction_plots$score_histogram)) {
        print(prediction_plots$score_histogram)
    }
    
    # 保存所有图表
    save_plots(prediction_plots, paths$output_plots, prefix = "prediction", 
               width = 12, height = 10)
    
    cat("\n✅ 可视化完成\n")
    
} else {
    cat("⚠️ 未找到预测结果，请先运行Cell 7\n")
}

In [ ]:
# ============================================
# Cell 9: 导出结果
# ============================================

if ("predicted.id" %in% colnames(xenium.obj@meta.data)) {
    
    # 主导出函数
    export_df <- export_results(
        xenium.obj = xenium.obj,
        flex_data.obj = flex_data.obj,
        paths = paths,
        params = params,
        mapping = reverse_mapping
    )
    
    # 显示前几行预览
    cat("\n📊 结果预览（前10行）:\n")
    print(head(export_df[, 1:min(6, ncol(export_df))], 10))
    
} else {
    cat("⚠️ 未找到预测结果，请先运行Cell 7\n")
}

In [ ]:
# ============================================
# Cell 10: 最终检查和质量报告
# ============================================

cat("\n" + paste(rep("⭐", 30), collapse = ""), "\n")
cat("✅ Pipeline 执行完成\n")
cat(paste(rep("⭐", 30), collapse = ""), "\n")

# 最终数据完整性检查
final_check <- list(
    flex_cells = ncol(flex_data.obj),
    xenium_cells = ncol(xenium.obj),
    predicted_cells = sum(!is.na(xenium.obj$predicted.id)),
    output_files = all(file.exists(c(paths$output_csv, paths$output_rds, paths$output_stats)))
)

cat("\n📊 最终统计:\n")
cat("  - Flex参考细胞数:", format_number(final_check$flex_cells), "\n")
cat("  - Xenium总细胞数:", format_number(final_check$xenium_cells), "\n")
cat("  - 预测细胞数:", format_number(final_check$predicted_cells), "\n")
cat("  - 输出文件完整性:", ifelse(final_check$output_files, "✅", "❌"), "\n")

# 内存使用报告
monitor_memory("最终")

cat("\n🎉 所有分析完成！\n")
cat("📂 输出文件位置:\n")
cat("  - 基础结果:", paths$output_csv, "\n")
cat("  - 完整结果:", paths$output_full_csv, "\n")
cat("  - Seurat对象:", paths$output_rds, "\n")
cat("  - 统计报告:", paths$output_stats, "\n")
cat("  - 可视化图表:", paths$output_plots, "\n")